In [ ]:
import numpy as np
import pandas as pd
import os
import csv
import ast
import seaborn as sns
from matplotlib import pyplot as plt 

from openfisca_france_indirect_taxation import FranceIndirectTaxationTaxBenefitSystem
from openfisca_france_indirect_taxation.surveys import SurveyScenario
from openfisca_france_indirect_taxation.utils import assets_directory, get_input_data_frame
from openfisca_france_indirect_taxation.examples.utils_example import (
    dataframe_by_group,
    df_weighted_average_grouped,
    wavg)

from openfisca_france_indirect_taxation.build_survey_data.utils import weighted_sum
from openfisca_france_indirect_taxation.Calage_consommation_bdf import get_inflators_by_year
from openfisca_france_indirect_taxation.Calage_revenus_bdf import calage_bdf_niveau_vie, compute_erfs_decile 
from openfisca_france_indirect_taxation.projects.TVA.Utils import weighted_quantiles, stacked_bar_plot, double_stacked_bar_plot, bootstrap_weighted_mean_by_decile

In [ ]:
output_path = 'C:/Users/veve1/OneDrive/Documents/IPP/Budget 2025 TVA/Figures_new'
output_path_data = 'C:/Users/veve1/OneDrive/Documents/IPP/Budget 2025 TVA/Figures_new/Data_new'

## Structure de la consommation par type de taux de TVA, par décile de niveau de vie

### Comparaison André Biotteau

In [ ]:
simulated_variables = ['depenses_tva_taux_plein',
'depenses_tva_taux_intermediaire',
'depenses_tva_taux_reduit',
'depenses_tva_taux_super_reduit',
'depenses_tva_exonere',
'depenses_totales',
'loyer_impute',
'tva_taux_plein',
 'tva_taux_intermediaire',
 'tva_taux_reduit',
 'tva_taux_super_reduit',
 'tva_total',
 'niveau_de_vie',
 'rev_disponible',
 'decile_rev_disponible',
 'rev_disp_yc_loyerimpute',
 'depenses_tot',
 'pondmen'] + ["poste_agrege_0{}".format(i) for i in range(1, 10)] + ["poste_agrege_{}".format(i) for i in range(10, 14)] 

In [ ]:
year = 2017
data_year = 2017
tax_benefit_system = FranceIndirectTaxationTaxBenefitSystem()
inflators_by_year = get_inflators_by_year(rebuild = True, year_range = range(2017, 2018), data_year = data_year)
inflation_kwargs = dict(inflator_by_variable = inflators_by_year[year])
survey_scenario = SurveyScenario.create(
    inflation_kwargs = inflation_kwargs,
    tax_benefit_system = tax_benefit_system,
    period = year,
    data_year = data_year
)

In [ ]:
structure_conso_decile = dataframe_by_group(survey_scenario, category = 'niveau_vie_decile', variables = simulated_variables, aggfunc = 'mean')
structure_conso_decile['poste_agrege_12'] += structure_conso_decile['poste_agrege_13']
structure_conso_decile.drop('poste_agrege_13', axis = 1, inplace = True)

In [ ]:
liste_postes_agreges = ["poste_agrege_0{}".format(i) for i in range(1, 10)] + ["poste_agrege_{}".format(i) for i in range(10, 13)]

### Structure de la consommation par poste agrégé

In [ ]:
label_postes_agreges = ['Alimentation', 'Alcools et tabacs', 'Habillement et chaussures', 'Logement', 'Ameublement et équipement ménager', 'Santé',
                        'Transports', 'Communication' , 'Loisirs et cultures' , 'Education', 'Restauration et hôtellerie', 'Services et biens divers']
liste_postes_agreges = ["poste_agrege_0{}".format(i) for i in range(1, 10)] + ["poste_agrege_{}".format(i) for i in range(10, 13)]

In [ ]:
for poste in liste_postes_agreges:
    structure_conso_decile['Part poste {}'.format(poste[-2:])] = structure_conso_decile[poste] / structure_conso_decile['depenses_tot']
structure_conso_decile.index = structure_conso_decile.index.astype(int)

In [ ]:
stacked_bar_plot(structure_conso_decile, 
                 variables = ["Part poste 0{}".format(i) for i in range(1, 10)] + ["Part poste {}".format(i) for i in range(10, 13)],
                 labels = label_postes_agreges,
                 title = 'Structure de la consommation par postes agrégés',
                 xlabel = 'Dixième de niveau de vie' ,
                 ylabel = '',
                 colors = list(sns.color_palette("Paired")),
                 savefig = True,
                 output_path = output_path,
                 outfile = 'Structure_conso_postes_agreges.pdf')

### Structure de la consommation par taux de TVA 

In [ ]:
structure_conso_decile[['depenses_tva_taux_plein' , 'depenses_tva_taux_intermediaire', 'depenses_tva_taux_reduit','depenses_tva_taux_super_reduit','depenses_tva_exonere','depenses_tot']]

In [ ]:
structure_conso_decile.loc[:,'Part TVA exonere'] = structure_conso_decile['depenses_tva_exonere'] / structure_conso_decile['depenses_tot']
structure_conso_decile.loc[:,'Part TVA taux super réduit'] = structure_conso_decile['depenses_tva_taux_super_reduit'] / structure_conso_decile['depenses_tot']
structure_conso_decile.loc[:,'Part TVA taux réduit'] = structure_conso_decile['depenses_tva_taux_reduit'] / structure_conso_decile['depenses_tot']
structure_conso_decile.loc[:,'Part TVA taux intermédiaire'] = structure_conso_decile['depenses_tva_taux_intermediaire'] / structure_conso_decile['depenses_tot']
structure_conso_decile.loc[:,'Part TVA taux plein'] = structure_conso_decile['depenses_tva_taux_plein'] / structure_conso_decile['depenses_tot']

In [ ]:
stacked_bar_plot(structure_conso_decile, 
                 variables = ['Part TVA taux plein','Part TVA taux intermédiaire','Part TVA taux réduit','Part TVA taux super réduit','Part TVA exonere'],
                 labels = ['Taux plein','Taux intermédiaire','Taux réduit','Taux super réduit','Taux exonere'],
                 title = 'Structure de la consommation par type de taux de TVA',
                 xlabel = 'Dixième de niveau de vie' ,
                 ylabel ='',
                 colors = list(sns.color_palette("Paired")),
                 savefig = True,
                 output_path = output_path,
                 outfile = 'Structure_conso_taux_tva_avec_exo.pdf')

In [ ]:
structure_conso_decile.to_csv(os.path.join(output_path_data, 'Structure_conso_decile_2017.csv'), index = True)

#### Hors taux éxonérés

In [ ]:
structure_conso_decile.loc[:,'Part TVA taux super réduit'] = structure_conso_decile['depenses_tva_taux_super_reduit'] / structure_conso_decile['depenses_totales']
structure_conso_decile.loc[:,'Part TVA taux réduit'] = structure_conso_decile['depenses_tva_taux_reduit'] / structure_conso_decile['depenses_totales']
structure_conso_decile.loc[:,'Part TVA taux intermédiaire'] = structure_conso_decile['depenses_tva_taux_intermediaire'] / structure_conso_decile['depenses_totales']
structure_conso_decile.loc[:,'Part TVA taux plein'] = structure_conso_decile['depenses_tva_taux_plein'] / structure_conso_decile['depenses_totales']

In [ ]:
stacked_bar_plot(structure_conso_decile, 
                 variables = ['Part TVA taux plein','Part TVA taux intermédiaire','Part TVA taux réduit','Part TVA taux super réduit'],
                 labels = ['Taux plein','Taux intermédiaire','Taux réduit','Taux super réduit'],
                 title = 'Structure de la consommation par type de taux de TVA',
                 xlabel = 'Dixième de niveau de vie' ,
                 ylabel ='',
                 colors = list(sns.color_palette("Paired")),
                 savefig = True,
                 output_path = output_path,
                 outfile = 'Structure_conso_taux_tva.pdf')

In [ ]:
total_graph = pd.DataFrame(structure_conso_decile[['depenses_tva_taux_plein' , 'depenses_tva_taux_intermediaire', 
          'depenses_tva_taux_reduit','depenses_tva_taux_super_reduit',
          'depenses_tva_exonere','depenses_tot', 'depenses_totales','tva_total']].mean()).T

In [ ]:
total_graph.loc[:,'Part TVA exonere'] = total_graph['depenses_tva_exonere'] / total_graph['depenses_tot']
total_graph.loc[:,'Part TVA taux super réduit'] = total_graph['depenses_tva_taux_super_reduit'] / total_graph['depenses_tot']
total_graph.loc[:,'Part TVA taux réduit'] = total_graph['depenses_tva_taux_reduit'] / total_graph['depenses_tot']
total_graph.loc[:,'Part TVA taux intermédiaire'] = total_graph['depenses_tva_taux_intermediaire'] / total_graph['depenses_tot']
total_graph.loc[:,'Part TVA taux plein'] = total_graph['depenses_tva_taux_plein'] / total_graph['depenses_tot']

In [ ]:
total_graph[['Part TVA exonere','Part TVA taux super réduit','Part TVA taux réduit','Part TVA taux intermédiaire','Part TVA taux plein']]

In [ ]:
total_graph.loc[:,'Part TVA taux super réduit'] = total_graph['depenses_tva_taux_super_reduit'] / total_graph['depenses_totales']
total_graph.loc[:,'Part TVA taux réduit'] = total_graph['depenses_tva_taux_reduit'] / total_graph['depenses_totales']
total_graph.loc[:,'Part TVA taux intermédiaire'] = total_graph['depenses_tva_taux_intermediaire'] / total_graph['depenses_totales']
total_graph.loc[:,'Part TVA taux plein'] = total_graph['depenses_tva_taux_plein'] / total_graph['depenses_totales']

In [ ]:
total_graph[['Part TVA taux super réduit','Part TVA taux réduit','Part TVA taux intermédiaire','Part TVA taux plein']]

## Comparaison Gesta (2025) Trésor-Eco

In [ ]:
year = 2022
data_year = 2017
tax_benefit_system = FranceIndirectTaxationTaxBenefitSystem()
inflators_by_year = get_inflators_by_year(rebuild = True, year_range = range(2022, 2023), data_year = data_year)
inflation_kwargs = dict(inflator_by_variable = inflators_by_year[year])
survey_scenario = SurveyScenario.create(
    inflation_kwargs = inflation_kwargs,
    tax_benefit_system = tax_benefit_system,
    period = year,
    data_year = data_year)

In [ ]:
depenses_totales =survey_scenario.compute_aggregate(variable = 'depenses_totales', use_baseline = True, period = year) 
depenses_taux_plein = survey_scenario.compute_aggregate(variable = 'depenses_tva_taux_plein', use_baseline = True, period = year) 
depenses_taux_intermediaire = survey_scenario.compute_aggregate(variable = 'depenses_tva_taux_intermediaire', use_baseline = True, period = year) 
depenses_taux_reduit = survey_scenario.compute_aggregate(variable = 'depenses_tva_taux_reduit', use_baseline = True, period = year) 
depenses_taux_super_reduit = survey_scenario.compute_aggregate(variable = 'depenses_tva_taux_super_reduit', use_baseline = True, period = year)

In [ ]:
structure_assiette_tva = pd.DataFrame({'depenses_tot': depenses_totales,
                                    'depenses_tot_tva_taux_plein': depenses_taux_plein,
                                    'depenses_tot_tva_taux_intermediaire': depenses_taux_intermediaire,
                                    'depenses_tot_tva_taux_reduit': depenses_taux_reduit,
                                    'depenses_tot_tva_taux_super_reduit': depenses_taux_super_reduit},index = [year])
liste_taux = ['plein','intermediaire', 'reduit', 'super_reduit']
for taux in liste_taux:
   structure_assiette_tva['Part_assiette_tva_taux_{}'.format(taux)] = structure_assiette_tva['depenses_tot_tva_taux_{}'.format(taux)] / structure_assiette_tva['depenses_tot'] * 100


In [ ]:
structure_assiette_tva 

In [ ]:
structure_assiette_tva.to_csv(os.path.join(output_path_data, 'Structure_assiette_tva_{}.csv'.format(year)), index = True)